# Supply Chain Data Cleaning & Analysis
Dataset: DataCo Smart Supply Chain for Big Data Analysis (Kaggle)

This notebook cleans a 180,519-row, 53-column global supply chain dataset, then exports a clean version for further analysis in SQL and Power BI.

## 1. Load the data

In [ ]:
import pandas as pd

df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')
print('Shape:', df.shape)
df.info()

## 2. Explore data quality issues
Check for missing values and duplicate rows before cleaning.

In [ ]:
nulls = df.isnull().sum()
print('Columns with missing values:')
print(nulls[nulls > 0])

print('\nDuplicate rows:', df.duplicated().sum())

## 3. Clean the data
- Drop columns with no analytical value: fully empty, near-empty, or masked placeholder fields (e.g. `Customer Email`/`Customer Password` are masked as `XXXXXXXXX` in the source data)
- Drop the small number of rows missing `Customer Lname` / `Customer Zipcode`
- Convert date columns to proper datetime types
- Sanity-check quantity and price columns for invalid (negative/zero) values

In [ ]:
rows_before = len(df)
cols_before = df.shape[1]

drop_cols = ['Product Description', 'Order Zipcode', 'Customer Email', 'Customer Password',
             'Customer Street', 'Product Image']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

df = df.dropna(subset=['Customer Lname', 'Customer Zipcode'])

df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'], errors='coerce')
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'], errors='coerce')

invalid_qty = (df['Order Item Quantity'] <= 0).sum()
invalid_price = (df['Product Price'] < 0).sum()

rows_after = len(df)
cols_after = df.shape[1]

print(f'Rows before: {rows_before}  ->  Rows after: {rows_after}')
print(f'Columns before: {cols_before}  ->  Columns after: {cols_after}')
print(f'Invalid quantity rows: {invalid_qty}')
print(f'Invalid price rows: {invalid_price}')
print('Remaining nulls:', df.isnull().sum().sum())

## 4. Export the cleaned dataset
This cleaned file is used for SQL querying and the Power BI dashboard.

In [ ]:
df.to_csv('DataCoSupplyChain_cleaned.csv', index=False)
print('Saved cleaned dataset:', df.shape)

## Summary

| Metric | Before | After |
|---|---|---|
| Rows | 180,519 | 180,508 |
| Columns | 53 | 47 |
| Missing values | present in 4 columns | 0 |

Next steps: queried with SQL (see `sql_queries.sql`) and visualized in Power BI (3-page dashboard: Data Quality, Insights, Trends & Status).